In [19]:
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage
import os
import base64
from ipywidgets import FileUpload
from IPython.display import display

# ==========================================
# 1. 基础配置与初始化
# ==========================================
load_dotenv(override=True)
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

# 换成视觉模型！
chat_model = init_chat_model(
    model="qwen3.8-flash",  # 👈 如果报错没权限，去百炼控制台开一下
    model_provider="openai",
    api_key=api_key,
    base_url=base_url,
    extra_body={"enable_thinking": False}
)

checkpointer = InMemorySaver()

# ==========================================
# 2. 定义数据结构与工具
# ==========================================
class RecipeInfo(BaseModel):
    """智能菜谱的输出规范"""
    name: str = Field(description="菜名")
    ingredients: list[str] = Field(description="所需食材列表")
    nutrition: Literal["高", "中", "低"] = Field(description="营养价值评估")
    difficulty: Literal["简单", "中等", "困难"] = Field(description="制作难度")
    steps: list[str] = Field(description="简短的制作步骤")
    reason: str = Field(description="推荐或创意的理由")

@tool
def search_recipe_db(ingredients: str) -> str:
    """根据食材搜索本地数据库中的真实菜谱。"""
    if "土豆" in ingredients and "西红柿" in ingredients:
        return "数据库命中：土豆西红柿炖牛肉（营养极高，难度中等，步骤：1.切块 2.炖煮 3.调味）"
    elif "鸡蛋" in ingredients:
        return "数据库命中：清炒鸡蛋（营养中等，难度简单，步骤：1.打散 2.热油 3.翻炒）"
    else:
        return "数据库中未找到相关菜谱，请发挥你的创意自行设计。"

system_prompt_with_tool = """
你是“小当家”，一位热情的AI大厨。
【工作流程】：
1. 用户给你食材后，你必须先调用 `search_recipe_db` 工具去查询真实菜谱。
2. 如果数据库找到了，请严格基于数据库的结果来填写菜谱信息。
3. 只有当数据库返回“未找到”时，你才能发挥创意自己编一个。
"""

# ==========================================
# 3. 组装 Agent
# ==========================================
agent_with_tool = create_agent(
    model=chat_model,
    system_prompt=system_prompt_with_tool,
    response_format=RecipeInfo,
    tools=[search_recipe_db],
    checkpointer=checkpointer
)

# ==========================================
# 4. 创建上传组件与核心处理函数
# ==========================================
uploader = FileUpload(
    accept='*',
    multiple=False,
    description='上传食材图片',
    button_style='success'
)
print("👇 请点击下方按钮上传一张食材图片，上传完毕后，再点一次左边的运行按钮(▶️)：")
display(uploader)



👇 请点击下方按钮上传一张食材图片，上传完毕后，再点一次左边的运行按钮(▶️)：


FileUpload(value=(), accept='*', button_style='success', description='上传食材图片')

In [20]:
def process_and_invoke(upload_widget):
    # 1. 安全检查
    if not upload_widget.value:
        print("⚠️ 你还没有上传图片哦！请先点击上面的按钮选择一张图。")
        return

    # 2. 读取图片并转码
    uploaded_file = upload_widget.value[0]
    file_name = uploaded_file['name']
    img_bytes = bytes(uploaded_file['content'])
    base64_str = base64.b64encode(img_bytes).decode('utf-8')
    print(f"\n✅ 成功读取图片：{file_name}")

    # 3. 构建消息
    multimodal_message = HumanMessage(
        content=[
            {"type": "text", "text": "帮我看看这张图里有什么食材？能做什么菜？"},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{base64_str}"}
            }
        ]
    )

    # 4. 调用模型 (注意这里使用 agent_with_tool)
    config: RunnableConfig = {"configurable": {"thread_id": "kitchen_session_1"}}
    print("🚀 正在呼叫 AI 大厨看图中...")

    # 👈 注意：调用的是 agent_with_tool，接收变量叫 response1
    response1 = agent_with_tool.invoke(
        {"messages": [multimodal_message]},
        config
    )

    # 5. 打印结果 (注意这里使用的是 response1)
    if "structured_response" in response1:
        recipe = response1["structured_response"]
        print(f"\n🍲 菜名：{recipe.name}")
        print(f"🥬 识别食材：{recipe.ingredients}")
        print(f"💪 营养：{recipe.nutrition} | 难度：{recipe.difficulty}")
        print(f"💡 推荐理由：{recipe.reason}")
    else:
        print("❌ 模型没有返回结构化结果，原始回复是：")
        print(response1["messages"][-1].content)

# ==========================================
# 5. 执行函数 (这行不能缩进)
# ==========================================
process_and_invoke(uploader)


✅ 成功读取图片：669767ba081c7be03048438d7612b2d8.jpeg
🚀 正在呼叫 AI 大厨看图中...

🍲 菜名：香煎土豆饼
🥬 识别食材：['土豆', '面粉', '鸡蛋', '盐', '黑胡椒', '食用油']
💪 营养：中 | 难度：简单
💡 推荐理由：由于数据库中未找到相关菜谱，为您创意设计了这道香煎土豆饼。它不仅能充分消耗图中的新鲜土豆，而且做法简单，口感外酥里嫩，非常适合作为日常快手菜或休闲小吃。
